**Notebook clientes**

In [1]:
!pip install pandas

In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/kevinnumanzor17/etl-seguros-pipeline/refs/heads/main/data/raw/clientes.csv"

df = pd.read_csv(url)

df.head()

,id_cliente,nombre,email,genero,fecha_nacimiento,ciudad,segmento
0,1,José Chávez Pérez,jose.chavez.perez1@gmail.com,Hombre,2011/11/24,SanMiguel,NaN
1,2,Daniela Rojas Ortiz,daniela.rojas.ortiz2@seguro.sv,NaN,01-02-80,Sta. Ana,NaN
2,3,Ricardo Cruz Flores,ricardo.cruz.flores3@example.com,Masculino,06-18-79,S. Miguel,NaN
3,4,María Pérez Pérez,maria.perez.perez4@correo.com,Masculino,1960-09-29,LaLibertad,REGULAR
4,5,Fernanda Chávez Rojas,fernanda.chavez.rojas5@seguro.sv,NaN,1945/08/02,San Salvador,Pyme


**Exploracion**

In [3]:
df.shape
df.columns
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id_cliente        3030 non-null   int64 
 1   nombre            3030 non-null   object
 2   email             3030 non-null   object
 3   genero            2435 non-null   object
 4   fecha_nacimiento  3030 non-null   object
 5   ciudad            3030 non-null   object
 6   segmento          2433 non-null   object
dtypes: int64(1), object(6)
memory usage: 165.8+ KB


,0
id_cliente,0
nombre,0
email,0
genero,595
fecha_nacimiento,0
ciudad,0
segmento,597


**Limpieza de datos**

In [4]:
clientes = df.copy()

# nombres de columnas
clientes.columns = clientes.columns.str.strip().str.lower()

# limpiar espacios en texto
for col in clientes.select_dtypes(include='object').columns:
    clientes[col] = clientes[col].astype(str).str.strip()

# convertir vacíos en null
clientes = clientes.replace(r'^\s*$', pd.NA, regex=True)

# normalizar email
clientes['email'] = clientes['email'].str.lower()

# convertir fecha
clientes['fecha_nacimiento'] = pd.to_datetime(clientes['fecha_nacimiento'], errors='coerce')

# eliminar duplicados
clientes = clientes.drop_duplicates()

**Separar válidos y rechazados**

In [5]:
validos = clientes[
    clientes['nombre'].notna() &
    clientes['email'].notna()
].copy()

rechazados = clientes[
    clientes['nombre'].isna() |
    clientes['email'].isna()
].copy()

**Motivo de rechazo**

In [6]:
def motivo(row):
    motivos = []

    if pd.isna(row['nombre']):
        motivos.append("nombre_vacio")

    if pd.isna(row['email']):
        motivos.append("email_vacio")

    return ",".join(motivos)

rechazados["motivo_rechazo"] = rechazados.apply(motivo, axis=1)

**exportar archivos**

In [7]:
validos.to_csv("clientes_curated.csv", index=False)
rechazados.to_csv("clientes_rejects.csv", index=False)